# PCA on food texture

A small dataset where five sensory and instrumental measurements are taken on each of 50 pastry samples: oil percentage, density, crispiness, fracture force, and hardness. The variables are correlated and measured on different scales, so PCA on the autoscaled data shows the dominant patterns in two components and lets us identify samples that do not look like the rest of the batch.

**Data.** `food-texture.csv` from [openmv.net](https://openmv.net/info/food-texture). The first column is a sample identifier; the remaining five are the measurements.

**Adapted from** the *Principal Component Analysis* chapter of the [Process Improvement using Data](https://learnche.org/pid) book (CC BY-SA 4.0).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.io as pio

from process_improve.multivariate.methods import PCA, MCUVScaler

pio.renderers.default = "notebook_connected"

## Load the data

In [ ]:
foods = pd.read_csv("https://openmv.net/file/food-texture.csv", index_col=0)
print(f"Shape: {foods.shape[0]} samples x {foods.shape[1]} variables")
foods.head()

In [ ]:
foods.describe().round(2)

The five variables span very different ranges (oil percentage is in the teens, density is in the thousands, the rest are unitless sensory scores). Without scaling, density would dominate the model purely through its variance.

## Preprocess and fit

In [ ]:
X = MCUVScaler().fit_transform(foods)

selection = PCA.select_n_components(X, max_components=4, cv=5)
print(f"Recommended A = {selection.n_components}")
selection.press_ratio.round(3)

In [ ]:
model = PCA(n_components=max(2, selection.n_components)).fit(X)
model.r2_cumulative_.round(3)

## Score and loading plots

With only five variables, the loading plot is more useful than for high-dimensional spectra: every loading carries a recognisable label, and we can read directly which sensory and instrumental measurements drive each component.

In [ ]:
model.score_plot()

In [ ]:
model.loading_plot()

## SPE and Hotelling's T squared

In [ ]:
model.spe_plot()

In [ ]:
model.t2_plot()

## Outlier explanation

Each flagged sample can be explained back in terms of the original five variables with `score_contributions`.

In [ ]:
outliers = model.detect_outliers(conf_level=0.95)
print(f"{len(outliers)} samples flagged")
for o in outliers[:3]:
    print(f"  {o['observation']}: {o['outlier_types']} (severity={o['severity']:.2f})")

In [ ]:
if outliers:
    worst = outliers[0]["observation"]
    contrib = model.score_contributions(model.scores_.loc[worst].values)

    fig, ax = plt.subplots(figsize=(6, 3))
    contrib.plot.bar(ax=ax, color="steelblue")
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_ylabel("Contribution")
    ax.set_title(f"Score contributions for sample {worst}")
    plt.tight_layout()
    plt.show()

## What to try next

- The food-texture dataset is small enough that you can recompute everything by hand and check intuition. Try changing the number of components and watch what happens to the score plot, the SPE limit, and the list of flagged samples.
- Compare the loading plot against domain knowledge: do the variables that cluster together in the loading plot also correlate strongly in `foods.corr()`?
- Once a property like consumer acceptance is added, switch to PLS to relate the texture variables to it.